# Quick Model Verification — LogisticGLM Only

**Purpose:** Minimal test to verify the pipeline works without PyTorch/GRU dependencies.

**Runtime:** ~30-60 seconds

**Configuration:**
- Model: LogisticGLM only (no GRU to avoid PyTorch issues)
- Training sizes: [50, 100]
- Bootstrap sizes: [25]
- Iterations: 2 per config
- **Total: 4 evaluations** (quick test)

If this works, GRU issues are due to PyTorch. If this also crashes, the issue is elsewhere.

In [ ]:
import sys, logging, traceback
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add rebuild directory to imports
sys.path.insert(0, '.')
logging.basicConfig(level=logging.WARNING)

print("Importing modules...")

from data_loader import (
    load_physionet_files,
    add_hours_until_sepsis,
    split_patients_by_status,
    get_rows_for_patients,
)
from bootstrap import BootstrapResampler
from models import LogisticGLM
from training import BootstrapEvaluator, extract_Xy

print("✓ All imports successful")

## 1. Configuration

In [ ]:
# MINIMAL configuration for quick test
TRAIN_SIZES = [50, 100]
BOOTSTRAP_SIZES = [25]
N_ITER = 2  # Just 2 iterations
RANDOM_STATE = 42

print(f"Config: {len(TRAIN_SIZES)} train × {len(BOOTSTRAP_SIZES)} boot × {N_ITER} iter = {len(TRAIN_SIZES) * len(BOOTSTRAP_SIZES) * N_ITER} evaluations")

## 2. Load Data

In [ ]:
print("Loading data...")
DATA_DIR = Path('../data/physionet_sepsis')

try:
    raw_df = load_physionet_files(DATA_DIR)
    print(f"✓ Loaded: {raw_df['patient_id'].nunique():,} patients, {len(raw_df):,} rows")
except Exception as e:
    print(f"✗ Data loading failed: {e}")
    traceback.print_exc()
    raise

print("Computing hours_until_sepsis...")
try:
    df = add_hours_until_sepsis(raw_df, keep_post_onset=True)
    n_septic = df['hours_until_sepsis'].notna().sum()
    print(f"✓ Computed: {n_septic:,} septic rows")
except Exception as e:
    print(f"✗ hours_until_sepsis failed: {e}")
    traceback.print_exc()
    raise

## 3. Test with One Config First

Before running the full loop, test a single config to identify the exact failure point.

In [ ]:
print("\n" + "="*70)
print("SINGLE CONFIG TEST")
print("="*70)

train_size = 50
boot_size = 25

print(f"\n[Step 1] Splitting patients (train={train_size})...")
try:
    train_pids, boot_pids = split_patients_by_status(
        df, n_train_patients=train_size,
        random_state=RANDOM_STATE, stratify_by_sepsis=True
    )
    print(f"✓ Split complete: {len(train_pids)} train, {len(boot_pids)} bootstrap")
except Exception as e:
    print(f"✗ Patient split failed: {e}")
    traceback.print_exc()
    raise

print(f"\n[Step 2] Getting training data...")
try:
    train_df = get_rows_for_patients(df, train_pids)
    print(f"✓ Training data: {len(train_df)} rows, {len(np.unique(train_pids))} unique patients")
except Exception as e:
    print(f"✗ Getting training data failed: {e}")
    traceback.print_exc()
    raise

print(f"\n[Step 3] Creating model...")
try:
    model = LogisticGLM(C=0.01)
    print(f"✓ Model created: {model.__class__.__name__}")
except Exception as e:
    print(f"✗ Model creation failed: {e}")
    traceback.print_exc()
    raise

print(f"\n[Step 4] Creating BootstrapEvaluator (this fits the model)...")
try:
    evaluator = BootstrapEvaluator(
        model=model,
        train_df=train_df,
        label_column='SepsisLabel',
        patient_id_column='patient_id'
    )
    print(f"✓ BootstrapEvaluator created and model fitted")
except Exception as e:
    print(f"✗ BootstrapEvaluator creation/model fit failed: {e}")
    traceback.print_exc()
    raise

print(f"\n[Step 5] Creating BootstrapResampler...")
try:
    resampler = BootstrapResampler(
        bootstrap_pool_patient_ids=boot_pids,
        full_df=df,
        n_iterations=N_ITER,
        bootstrap_sample_size=boot_size,
        random_state=RANDOM_STATE
    )
    print(f"✓ BootstrapResampler created")
except Exception as e:
    print(f"✗ BootstrapResampler creation failed: {e}")
    traceback.print_exc()
    raise

print(f"\n[Step 6] Generating first bootstrap sample...")
try:
    _, boot_df = resampler.generate_iteration(0)
    print(f"✓ Bootstrap sample generated: {len(boot_df)} rows, {boot_df['patient_id'].nunique()} unique patients")
except Exception as e:
    print(f"✗ Bootstrap generation failed: {e}")
    traceback.print_exc()
    raise

print(f"\n[Step 7] Evaluating on bootstrap sample...")
try:
    m = evaluator.evaluate_iteration(
        boot_df, 0,
        compute_per_group=True,
        group_column='Gender'
    )
    print(f"✓ Evaluation complete")
    print(f"  Utility: {m.get('utility', np.nan):.4f}")
    print(f"  AUROC: {m.get('auroc', np.nan):.4f}")
    print(f"  Recall: {m.get('recall', np.nan):.4f}")
except Exception as e:
    print(f"✗ Evaluation failed: {e}")
    traceback.print_exc()
    raise

print("\n" + "="*70)
print("✓ SINGLE CONFIG TEST PASSED")
print("="*70)

## 4. Run All Configurations

Now that we know the pipeline works, run the full (minimal) test.

In [ ]:
print("\n" + "="*70)
print("FULL EVALUATION")
print("="*70)

results = []
config_num = 0
total_configs = len(TRAIN_SIZES) * len(BOOTSTRAP_SIZES) * N_ITER

for train_size in TRAIN_SIZES:
    print(f"\n  Train size: {train_size}")
    
    # Split patients
    train_pids, boot_pids = split_patients_by_status(
        df, n_train_patients=train_size,
        random_state=RANDOM_STATE, stratify_by_sepsis=True
    )
    train_df = get_rows_for_patients(df, train_pids)
    
    for boot_size in BOOTSTRAP_SIZES:
        print(f"    Boot size: {boot_size}")
        
        # Create model and evaluator
        model = LogisticGLM(C=0.01)
        evaluator = BootstrapEvaluator(
            model=model,
            train_df=train_df,
            label_column='SepsisLabel',
            patient_id_column='patient_id'
        )
        
        # Bootstrap resampler
        resampler = BootstrapResampler(
            bootstrap_pool_patient_ids=boot_pids,
            full_df=df,
            n_iterations=N_ITER,
            bootstrap_sample_size=boot_size,
            random_state=RANDOM_STATE
        )
        
        # Evaluate on each bootstrap sample
        for iter_idx in range(N_ITER):
            config_num += 1
            print(f"      [{config_num:2d}/{total_configs}] Iteration {iter_idx+1}... ", end='', flush=True)
            
            try:
                _, boot_df = resampler.generate_iteration(iter_idx)
                
                m = evaluator.evaluate_iteration(
                    boot_df, iter_idx,
                    compute_per_group=True,
                    group_column='Gender'
                )
                
                # Extract utility by group
                utility_female = np.nan
                utility_male = np.nan
                if 'per_group' in m:
                    if 0 in m['per_group']:
                        utility_female = m['per_group'][0].get('utility', np.nan)
                    if 1 in m['per_group']:
                        utility_male = m['per_group'][1].get('utility', np.nan)
                
                results.append({
                    'train_size': train_size,
                    'boot_size': boot_size,
                    'iteration': iter_idx,
                    'utility': m.get('utility', np.nan),
                    'auroc': m.get('auroc', np.nan),
                    'recall': m.get('recall', np.nan),
                    'f1': m.get('f1', np.nan),
                    'accuracy': m.get('accuracy', np.nan),
                    'utility_female': utility_female,
                    'utility_male': utility_male,
                })
                
                print(f"✓ Utility: {m.get('utility', np.nan):.4f}")
                
            except Exception as e:
                print(f"✗ ERROR: {str(e)[:60]}")
                results.append({
                    'train_size': train_size,
                    'boot_size': boot_size,
                    'iteration': iter_idx,
                    'utility': np.nan,
                    'auroc': np.nan,
                    'recall': np.nan,
                    'f1': np.nan,
                    'accuracy': np.nan,
                    'utility_female': np.nan,
                    'utility_male': np.nan,
                })
                traceback.print_exc()

print(f"\n\n{'='*70}")
print(f"EVALUATION COMPLETE: {len(results)} results")
print(f"{'='*70}")

## 5. Results Summary

In [ ]:
results_df = pd.DataFrame(results)

print(f"\nTotal results: {len(results_df)}")
print(f"Valid (non-NaN utility): {results_df['utility'].notna().sum()}")
print(f"\nResults:\n")
print(results_df.round(4).to_string(index=False))

# Summary statistics
if results_df['utility'].notna().any():
    print(f"\n\nUtility Stats:")
    print(f"  Mean:  {results_df['utility'].mean():.4f}")
    print(f"  Std:   {results_df['utility'].std():.4f}")
    print(f"  Min:   {results_df['utility'].min():.4f}")
    print(f"  Max:   {results_df['utility'].max():.4f}")

## 6. Save Results

In [ ]:
results_df.to_csv('utility_logisticglm_quick.csv', index=False)
print("✓ Saved: utility_logisticglm_quick.csv")

print("\n" + "="*70)
print("SUCCESS: Pipeline works with LogisticGLM")
print("="*70)
print(f"\nIf you need GRU (RNN) models:")
print(f"  1. Install PyTorch: pip install torch")
print(f"  2. Run test_utility_models_minimal.ipynb")
print(f"\nIf you want full evaluation:")
print(f"  1. Run test_utility_at_scales.ipynb for LogisticGLM only")
print(f"  2. Or run test_utility_model_comparison.ipynb for all 3 models")